# 00 — Load & clean the SBA 7(a) data

**Plain English:** We read the real U.S. Small Business Administration 7(a) loan files, tidy them up, and check the data looks sensible before any analysis.

Each row is one approved small-business loan. We:
- parse the dates (approval, paid-in-full, charge-off),
- standardise the `LoanStatus` codes (e.g. `P I F` = *paid in full*, `CHGOFF` = *charged off / defaulted*),
- map each loan's 6-digit NAICS code to a plain industry name, and
- drop loans that were cancelled or only committed (they never funded, so they carry no real exposure).

**Key term — charge-off:** when a lender writes a loan off as a loss. We treat charge-off as *default* throughout this project.

In [1]:
%matplotlib inline
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
pd.set_option('display.max_columns', 40, 'display.width', 200)
from src.config import load_config, TABLES_DIR
CFG = load_config()

In [2]:
from src.data_loader import load_clean, data_quality_summary
loans = load_clean(config=CFG)
loans.shape

[2026-06-12 11:24:45] [INFO    ] [data_loader] Loading 2 SBA input file(s): ['foia-7a-fy2000-fy2009-asof-260331.csv', 'foia-7a-fy2010-fy2019-asof-260331.csv']


[2026-06-12 11:24:49] [INFO    ] [data_loader]   foia-7a-fy2000-fy2009-asof-260331.csv -> 690333 rows


[2026-06-12 11:24:51] [INFO    ] [data_loader]   foia-7a-fy2010-fy2019-asof-260331.csv -> 545751 rows


[2026-06-12 11:24:51] [INFO    ] [data_loader] Combined raw frame: 1236084 rows


[2026-06-12 11:24:54] [INFO    ] [data_loader] Dropped 149065 never-funded loans (['CANCLD', 'COMMIT']); 1087019 funded loans remain


(1087019, 22)

### Result: data-quality / coverage summary
One row per check. This is the table we save for this notebook.

In [3]:
dq = data_quality_summary(loans, config=CFG)
dq.to_csv(TABLES_DIR / '00_data_quality_summary.csv', index=False)
dq

,metric,value
0,Funded loans,"1,087,019"
1,Approval FY range,2000-2019
2,Total gross approval ($),"287,809,010,585"
3,Charged-off loans (count),"132,662"
4,Charge-off rate (count),12.20%
5,Charge-off rate ($),4.92%
6,Distinct NAICS sectors,21
7,Distinct borrower states,60
8,Distinct lenders,"4,221"
9,Missing NAICS sector,"16,922"


**Read-out:** ~1.09M funded 7(a) loans, fiscal years 2000–2019. The overall charge-off rate is ~12% by loan count but only ~5% by dollars — larger loans default less often, a pattern we quantify in notebook 03.